# Structural Damage Recognition — InceptionV3 Transfer Learning

**PHI-Net Task 2: Damage State Classification**

This notebook trains an InceptionV3 model with transfer learning on Google Colab GPU.

## Prerequisites
1. Upload these files to Google Drive folder `PHI-Net-Dataset/`:
   - `task2_X_train.npy` (~6.62 GiB)
   - `task2_y_train.npy` (~92 KiB)
   - `task2_X_test.npy` (~838 MiB)
   - `task2_y_test.npy` (~12 KiB)
2. Select **GPU runtime**: Runtime → Change runtime type → T4 GPU
3. Run cells in order

## 1. Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIGURE THESE PATHS ──────────────────────────────────────────
DRIVE_DATASET_DIR = '/content/drive/MyDrive/PHI-Net-Dataset'
DRIVE_OUTPUT_DIR  = '/content/drive/MyDrive/PHI-Net-Output'
LOCAL_STAGING_DIR = '/content/dataset'
# ───────────────────────────────────────────────────────────────────

import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_STAGING_DIR, exist_ok=True)
print(f'Drive dataset dir: {DRIVE_DATASET_DIR}')
print(f'Drive output dir:  {DRIVE_OUTPUT_DIR}')

## 2. Check Resources & Verify Dataset

In [ ]:
import shutil
import subprocess
import numpy as np

# Check GPU
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                           '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu_info.stdout.strip() if gpu_info.returncode == 0 else 'No GPU detected')

# Check RAM
import psutil
mem = psutil.virtual_memory()
print(f'RAM: {mem.total / 1e9:.1f} GB total, {mem.available / 1e9:.1f} GB available')

# Check disk space
for path in ['/content', DRIVE_DATASET_DIR]:
    try:
        total, used, free = shutil.disk_usage(path)
        print(f'Disk ({path}): {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB')
    except:
        print(f'Disk ({path}): not accessible')

# Verify dataset files
REQUIRED_FILES = [
    'task2_X_train.npy', 'task2_y_train.npy',
    'task2_X_test.npy', 'task2_y_test.npy'
]

print('\n--- Dataset Files ---')
all_present = True
for f in REQUIRED_FILES:
    path = os.path.join(DRIVE_DATASET_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f'  ✅ {f}: {size_mb:.1f} MiB')
    else:
        print(f'  ❌ {f}: NOT FOUND')
        all_present = False

if not all_present:
    raise FileNotFoundError(
        f'Missing dataset files in {DRIVE_DATASET_DIR}. '
        'Upload all 4 .npy files to your Google Drive folder.'
    )

## 3. Stage Dataset to Local Disk (if space allows)

In [ ]:
import time

REQUIRED_SPACE_GB = 8.5  # ~7.5 GB for files + buffer

_, _, free = shutil.disk_usage('/content')
free_gb = free / (1024**3)

USE_LOCAL = free_gb >= REQUIRED_SPACE_GB

if USE_LOCAL:
    print(f'Local disk has {free_gb:.1f} GB free. Staging files locally...')
    for f in REQUIRED_FILES:
        src = os.path.join(DRIVE_DATASET_DIR, f)
        dst = os.path.join(LOCAL_STAGING_DIR, f)
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            print(f'  {f}: already staged ✅')
            continue
        print(f'  Copying {f} ({os.path.getsize(src)/(1024**2):.0f} MiB)...')
        start = time.time()
        # Use filesystem copy (not Python read/write) to avoid RAM bloat
        shutil.copy2(src, dst)
        elapsed = time.time() - start
        print(f'    Done in {elapsed:.1f}s ✅')
    DATA_DIR = LOCAL_STAGING_DIR
    print(f'\nUsing LOCAL staging: {DATA_DIR}')
else:
    print(f'⚠️  Local disk only has {free_gb:.1f} GB free (need {REQUIRED_SPACE_GB} GB).')
    print('Falling back to direct Drive access (slower random reads).')
    DATA_DIR = DRIVE_DATASET_DIR

print(f'\nDATA_DIR = {DATA_DIR}')

## 4. Load & Inspect Dataset

In [ ]:
# Load with memory mapping
X_train = np.load(os.path.join(DATA_DIR, 'task2_X_train.npy'), mmap_mode='r', allow_pickle=False)
y_train = np.load(os.path.join(DATA_DIR, 'task2_y_train.npy'), allow_pickle=False)
X_test  = np.load(os.path.join(DATA_DIR, 'task2_X_test.npy'), mmap_mode='r', allow_pickle=False)
y_test  = np.load(os.path.join(DATA_DIR, 'task2_y_test.npy'), allow_pickle=False)

print(f'X_train: {X_train.shape}, {X_train.dtype}')
print(f'y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, {X_test.dtype}')
print(f'y_test:  {y_test.shape}')

y_train_cls = np.argmax(y_train, axis=1)
y_test_cls  = np.argmax(y_test, axis=1)

print(f'\nTrain — Damaged: {(y_train_cls==0).sum()}, Undamaged: {(y_train_cls==1).sum()}')
print(f'Test  — Damaged: {(y_test_cls==0).sum()}, Undamaged: {(y_test_cls==1).sum()}')

## 5. Preprocessing Functions

In [ ]:
MEAN_BGR = np.array([103.939, 116.779, 123.68], dtype=np.float32)

def preprocess_batch(batch_bgr):
    """Restore Caffe BGR → RGB, normalise to [-1, 1] for InceptionV3."""
    batch_rgb = (np.asarray(batch_bgr, dtype=np.float32) + MEAN_BGR)[:, :, :, ::-1]
    batch_rgb = np.clip(batch_rgb, 0.0, 255.0)
    return (batch_rgb / 127.5) - 1.0

def restore_for_viz(images):
    """Reverse preprocessing for matplotlib display."""
    imgs = np.asarray(images, dtype=np.float32) + MEAN_BGR
    imgs = np.clip(imgs, 0, 255).astype(np.uint8)
    return imgs[..., ::-1]  # BGR→RGB

# Quick sanity check
test_batch = preprocess_batch(X_test[:4])
print(f'Preprocessed batch range: [{test_batch.min():.3f}, {test_batch.max():.3f}]')
assert test_batch.min() >= -1.01 and test_batch.max() <= 1.01, 'Preprocessing range error!'

In [ ]:
import matplotlib.pyplot as plt

# Visualise sample images
rng = np.random.default_rng(42)
d_idx = rng.choice(np.where(y_test_cls == 0)[0], 4, replace=False)
u_idx = rng.choice(np.where(y_test_cls == 1)[0], 4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, idx in enumerate(d_idx):
    axes[0, i].imshow(restore_for_viz(X_test[idx:idx+1])[0])
    axes[0, i].set_title(f'Damaged (idx {idx})', fontweight='bold')
    axes[0, i].axis('off')
for i, idx in enumerate(u_idx):
    axes[1, i].imshow(restore_for_viz(X_test[idx:idx+1])[0])
    axes[1, i].set_title(f'Undamaged (idx {idx})', fontweight='bold')
    axes[1, i].axis('off')
plt.suptitle('Sample Test Images', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Stratified Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRACTION = 0.2

all_indices = np.arange(len(y_train))
train_idx, val_idx = train_test_split(
    all_indices, test_size=VAL_FRACTION, random_state=SEED, stratify=y_train_cls
)
train_idx.sort()
val_idx.sort()

print(f'Train: {len(train_idx)} samples')
print(f'Val:   {len(val_idx)} samples')
print(f'Train class dist: Damaged={np.sum(y_train_cls[train_idx]==0)}, Undamaged={np.sum(y_train_cls[train_idx]==1)}')
print(f'Val   class dist: Damaged={np.sum(y_train_cls[val_idx]==0)}, Undamaged={np.sum(y_train_cls[val_idx]==1)}')

## 7. Batch Generator

In [ ]:
import tensorflow as tf

class ColabBatchGenerator(tf.keras.utils.Sequence):
    """Memory-efficient batch generator for mmap arrays."""
    
    def __init__(self, X_mmap, y_labels, indices, batch_size=32, shuffle=True, seed=42):
        self.X_mmap = X_mmap
        self.y_labels = y_labels
        self.indices = np.array(indices)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))
    
    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_idx = self.indices[start:end]
        X_batch = preprocess_batch(self.X_mmap[batch_idx])
        y_batch = self.y_labels[batch_idx]
        return X_batch, y_batch
    
    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

# ── Configuration ──
BATCH_SIZE = 32

train_gen = ColabBatchGenerator(X_train, y_train, train_idx, batch_size=BATCH_SIZE, shuffle=True)
val_gen   = ColabBatchGenerator(X_train, y_train, val_idx,   batch_size=BATCH_SIZE, shuffle=False)

# Smoke test
X_b, y_b = train_gen[0]
print(f'Batch shape: X={X_b.shape}, y={y_b.shape}')
print(f'Batch X range: [{X_b.min():.3f}, {X_b.max():.3f}]')
print(f'Train batches: {len(train_gen)}, Val batches: {len(val_gen)}')

## 8. Build InceptionV3 Model

In [ ]:
# Build InceptionV3 transfer learning model
base_model = tf.keras.applications.InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(2, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='InceptionV3_Damage')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 9. Phase 1 — Train Frozen Head

In [ ]:
EPOCHS_FROZEN = 15

# Checkpoint and output dirs on Drive
CHECKPOINT_DIR = os.path.join(DRIVE_OUTPUT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

callbacks_frozen = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(CHECKPOINT_DIR, 'frozen_best.keras'),
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
]

print(f'Training frozen head for up to {EPOCHS_FROZEN} epochs...')
history_frozen = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FROZEN,
    callbacks=callbacks_frozen,
    verbose=1
)

## 10. Phase 2 — Fine-tune Top Layers

In [ ]:
EPOCHS_FINETUNE = 10
FINETUNE_LAYERS = 30

# Unfreeze top layers
base_model.trainable = True
for layer in base_model.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfroze top {trainable_count} layers of InceptionV3')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_finetune = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(CHECKPOINT_DIR, 'finetune_best.keras'),
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1
    ),
]

total_epochs = EPOCHS_FROZEN + EPOCHS_FINETUNE
initial_epoch = len(history_frozen.history['loss'])

print(f'\nFine-tuning for up to {EPOCHS_FINETUNE} more epochs...')
history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    callbacks=callbacks_finetune,
    verbose=1
)

## 11. Save Final Model & Training History

In [ ]:
import json

# Save model to Drive
MODEL_DIR = os.path.join(DRIVE_OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, 'mobilenet_v2_damage_classifier.keras')
model.save(model_path)
print(f'✅ Model saved to: {model_path}')

# Merge and save training history
full_history = {}
for key in history_frozen.history:
    full_history[key] = history_frozen.history[key] + history_finetune.history.get(key, [])

METRICS_DIR = os.path.join(DRIVE_OUTPUT_DIR, 'metrics')
os.makedirs(METRICS_DIR, exist_ok=True)

with open(os.path.join(METRICS_DIR, 'training_history.json'), 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in full_history.items()}, f, indent=2)

print('✅ Training history saved')

## 12. Plot Training Curves

In [ ]:
PLOTS_DIR = os.path.join(DRIVE_OUTPUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(full_history['loss'], label='Train')
ax1.plot(full_history['val_loss'], label='Validation')
ax1.axvline(x=initial_epoch-0.5, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(full_history['accuracy'], label='Train')
ax2.plot(full_history['val_accuracy'], label='Validation')
ax2.axvline(x=initial_epoch-0.5, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 13. Evaluate on Official Test Set

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)

# Predict on test set in batches
test_gen = ColabBatchGenerator(X_test, y_test, np.arange(len(y_test)),
                               batch_size=32, shuffle=False)
test_probs = model.predict(test_gen, verbose=1)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(test_probs, axis=1)

# Metrics
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
auc = roc_auc_score((y_true == 0).astype(int), test_probs[:, 0])

print(f'\n{"="*50}')
print(f'Official Test Accuracy: {acc*100:.2f}%')
print(f'ROC-AUC (Damaged):      {auc:.4f}')
print(f'{"="*50}')

report = classification_report(y_true, y_pred, target_names=['Damaged', 'Undamaged'])
report_dict = classification_report(y_true, y_pred, target_names=['Damaged', 'Undamaged'], output_dict=True)
print(f'\n{report}')
print(f'Confusion Matrix:\n{cm}')

# Save metrics
with open(os.path.join(METRICS_DIR, 'test_metrics.json'), 'w') as f:
    json.dump({
        'accuracy': float(acc),
        'roc_auc_damaged': float(auc),
        'confusion_matrix': cm.tolist(),
        'classification_report': report_dict
    }, f, indent=2)
with open(os.path.join(METRICS_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

print(f'\n✅ Metrics saved to {METRICS_DIR}')

## 14. Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
im = ax1.imshow(cm, cmap=plt.cm.Blues)
ax1.set_xticks([0, 1])
ax1.set_yticks([0, 1])
ax1.set_xticklabels(['Damaged', 'Undamaged'])
ax1.set_yticklabels(['Damaged', 'Undamaged'])
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix')
thresh = cm.max() / 2.
for i in range(2):
    for j in range(2):
        ax1.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > thresh else 'black',
                 fontsize=16, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve((y_true == 0).astype(int), test_probs[:, 0])
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve (Damaged Class)')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'eval_plots.png'), dpi=150)
plt.show()

## 15. Error Analysis

In [ ]:
LABEL_NAMES = {0: 'Damaged', 1: 'Undamaged'}

# Find misclassifications
fp_idx = np.where((y_true == 1) & (y_pred == 0))[0]  # Undamaged predicted as Damaged
fn_idx = np.where((y_true == 0) & (y_pred == 1))[0]  # Damaged predicted as Undamaged
tp_idx = np.where((y_true == 0) & (y_pred == 0))[0]  # Correctly Damaged
tn_idx = np.where((y_true == 1) & (y_pred == 1))[0]  # Correctly Undamaged

print(f'True Positives (Damaged correct): {len(tp_idx)}')
print(f'True Negatives (Undamaged correct): {len(tn_idx)}')
print(f'False Positives (Undamaged→Damaged): {len(fp_idx)}')
print(f'False Negatives (Damaged→Undamaged): {len(fn_idx)}')

# Plot examples
categories = [
    ('False Positives (Undamaged → Damaged)', fp_idx),
    ('False Negatives (Damaged → Undamaged)', fn_idx),
    ('True Positives (Damaged ✓)', tp_idx),
    ('True Negatives (Undamaged ✓)', tn_idx),
]

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for row, (title, indices) in enumerate(categories):
    selected = indices[:5] if len(indices) >= 5 else indices
    for col in range(5):
        ax = axes[row, col]
        if col < len(selected):
            idx = selected[col]
            img = restore_for_viz(X_test[idx:idx+1])[0]
            ax.imshow(img)
            conf = test_probs[idx][y_pred[idx]]
            ax.set_title(f'True: {LABEL_NAMES[y_true[idx]]}\n'
                        f'Pred: {LABEL_NAMES[y_pred[idx]]} ({conf:.1%})',
                        fontsize=9)
        ax.axis('off')
    axes[row, 0].set_ylabel(title, fontsize=10, rotation=0, labelpad=120, va='center')

plt.suptitle('Error Analysis — Test Set Predictions', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'error_analysis.png'), dpi=150)
plt.show()

## 16. Download Model for Local Inference

Copy the saved model from Google Drive to your local `models/` directory
to use with the Streamlit demo app.

In [ ]:
print(f'Model saved at: {model_path}')
print(f'Metrics at:     {METRICS_DIR}')
print(f'Plots at:       {PLOTS_DIR}')
print()
print('To use with the Streamlit app:')
print('  1. Download the model from Google Drive')
print('  2. Place it in: models/mobilenet_v2_damage_classifier.keras')
print('  3. Run: streamlit run app.py')